# GEPA + FewShot — Experiment Notebook

End-to-end walkthrough:
1. Install dependencies
2. Deploy a local SGLang model
3. Run experiments (GEPA, GEPAFewShot, MIPROv2)
4. Analyze and compare results

> **Note:** Steps 2–3 assume an NVIDIA GPU and a running SGLang server.  
> On a CPU-only machine, point `API_BASE` at an OpenAI-compatible remote endpoint instead.

## 1. Install dependencies

In [ ]:
# Install from local repo (editable) + extra deps
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '..', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'gepa', 'scikit-learn', 'psutil', 'matplotlib', 'pandas', '-q'], check=True)
print('Dependencies installed.')

## 2. Deploy local SGLang model

Skip this cell if your model is already served or you are using a remote API endpoint.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from remote_setup.utils import deploy_sglang_model, is_server_up

MODEL   = 'meta-llama/Llama-3.2-3B-Instruct'
PORT    = 30000
API_BASE = f'http://localhost:{PORT}/v1'

if is_server_up(port=PORT):
    print(f'SGLang server already running on port {PORT}.')
else:
    print(f'Deploying {MODEL} ...')
    deploy_sglang_model(model_path=MODEL, port=PORT)
    print('Server ready.')

## 3. Configure DSPy

In [ ]:
import dspy

dspy.settings.experimental = True

lm = dspy.LM(MODEL, api_base=API_BASE, api_key='local')
dspy.configure(lm=lm)
print('DSPy configured with:', MODEL)

## 4. Load dataset

Choose `DATASET = 'gsm8k'` or `'iris'`.

In [ ]:
DATASET    = 'gsm8k'   # 'gsm8k' | 'iris'
TRAIN_SIZE = 200
VAL_SIZE   = 100
TEST_SIZE  = 300

if DATASET == 'gsm8k':
    from dspy.datasets.gsm8k import GSM8K, gsm8k_metric
    dataset  = GSM8K()
    trainset = [x.with_inputs('question') for x in dataset.train][:TRAIN_SIZE]
    valset   = [x.with_inputs('question') for x in dataset.dev][:VAL_SIZE]
    testset  = [x.with_inputs('question') for x in dataset.test][:TEST_SIZE]
    metric   = gsm8k_metric

elif DATASET == 'iris':
    from dspy.datasets.iris import IrisDataset
    dataset  = IrisDataset(seed=0)
    trainset, valset, testset = dataset.get_data_splits()
    trainset = trainset[:TRAIN_SIZE]
    valset   = valset[:VAL_SIZE]
    testset  = testset[:TEST_SIZE]
    metric   = dspy.evaluate.answer_exact_match

print(f'{DATASET}: {len(trainset)} train / {len(valset)} val / {len(testset)} test')

## 5. Build student program

In [ ]:
from experiments.programs import CoT, IrisProgram

student = CoT() if DATASET == 'gsm8k' else IrisProgram()
print('Student:', student)

## 6. Run experiments

Runs GEPA, GEPAFewShot, and MIPROv2 sequentially with the `light` budget preset.
Results are printed inline and saved to `experiments/logs/`.

In [ ]:
import subprocess, sys, os

OPTIMIZERS = ['gepa', 'gepa_fewshot', 'miprov2']
AUTO       = 'light'
LOG_DIR    = 'experiments/logs'

for optimizer in OPTIMIZERS:
    print(f'\n{"="*50}')
    print(f'Running: {optimizer.upper()} on {DATASET}')
    print('='*50)
    cmd = [
        sys.executable, 'experiments/run_experiment.py',
        '--dataset',   DATASET,
        '--optimizer', optimizer,
        '--model',     MODEL,
        '--auto',      AUTO,
        '--log-dir',   LOG_DIR,
        '--train-size', str(TRAIN_SIZE),
        '--val-size',   str(VAL_SIZE),
        '--test-size',  str(TEST_SIZE),
    ]
    result = subprocess.run(cmd, capture_output=False, text=True, cwd='..')
    if result.returncode != 0:
        print(f'ERROR: {optimizer} run failed.')

## 7. Analyze results

In [ ]:
from experiments.analyze_results import load_runs, print_summary, plot_score_comparison

records = load_runs('experiments/logs', dataset_filter=DATASET)
print_summary(records)

In [ ]:
%matplotlib inline
plot_score_comparison(records, plot_dir='experiments/plots')

## 8. Inspect optimized instructions

In [ ]:
for r in records:
    print(f"\n{'='*60}")
    print(f"Run : {r['run_tag']}")
    print(f"Score: {r['test_score']:.4f}  |  Demos: {r['n_demos']}")
    for name, instr in r['optimized_instructions'].items():
        print(f"  [{name}]\n  {instr[:400]}")